# PCBuildAR — train the PC parts detector

Trains a YOLO11 detector on Roboflow Universe data and exports an ONNX that runs **on the phone**, offline.

Three datasets, pooled into one (all CC BY 4.0):

| Project | Images | Classes | Note |
|---|---|---|---|
| `james-manalili/pc-parts-5uy7m` | 1,369 | 12 | mostly parts inside PCs; sets the class list |
| `cas-pc-project/pc-parts-detection-yajmi` | 408 | same 12 | |
| `mynhungs-workspace/pc-detection-hu8q2` | 5,297 | 7 | many single-part, listing-style photos — closest to pointing a phone at a part |

The third project names its classes differently (`CPU_Fan`, `DISK`, `Mainboard`, `RAM`…). The merge cell renames
them onto the first project's names (`CLASS_ALIASES`), so the model keeps **the same 12 classes in the same order**
and nothing in the Unity project has to change.

**Known weakness of the third set:** it only labels 7 kinds of part, and its motherboards are boxed in some photos and
not in others. Where it shows an unlabelled motherboard, front panel or rear I/O, the model is taught "that's
background". The comparison cell scores each source's validation set separately so you can see whether that cost
more than the extra photos gained — trust those numbers, not the overall mAP.

**The bigger win is still your own photos.** Add a project of your own to `SOURCES` — step-by-step in
**`SHOOT_YOUR_OWN_DATASET.md`**, next to this notebook.

**Before you start:** Runtime ▸ Change runtime type ▸ **T4 GPU**. With ~7,000 images and `yolo11s`, training takes
roughly **2 hours** — long enough that a free Colab session can drop, so the Drive cell keeps the run safe and
lets you resume.

The last cell downloads two files. Put **both** in `Assets/_Project/ML/Models/`: `best.onnx` and
`labels_pcparts_roboflow.txt`.

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics roboflow

In [ ]:
# KEEP THE RUN SAFE (recommended): training writes straight into Google Drive, so a dropped Colab session loses
# nothing — re-run the cells and resume (see the training cell). Set USE_DRIVE = False to keep it all in Colab.
import os

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/PCBuildAR"
else:
    OUT = "/content/PCBuildAR"
os.makedirs(OUT, exist_ok=True)
print("results will be kept in", OUT)

In [ ]:
# Your free key: roboflow.com -> Settings -> API Keys. Keep it out of screenshots and reports.
from roboflow import Roboflow

API_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY"

# The FIRST project sets the class names and their order (the app's labels file). Add your own project here too.
SOURCES = [
    ("james-manalili", "pc-parts-5uy7m"),            # 1,369 images, 12 classes
    ("cas-pc-project", "pc-parts-detection-yajmi"),  #   408 images, same 12 classes
    ("mynhungs-workspace", "pc-detection-hu8q2"),     # 5,297 images, 7 classes (renamed in the merge cell)
]

rf = Roboflow(api_key=API_KEY)
datasets = []

for workspace, name in SOURCES:
    project = rf.workspace(workspace).project(name)
    try:
        numbers = sorted(int(str(v.version).split("/")[-1]) for v in project.versions())
        latest = numbers[-1]
        print(f"{name}: versions {numbers}")
    except Exception as e:
        print(f"{name}: could not list versions ({e}) - falling back to version 1")
        latest = 1
    print(f"{name}: downloading version {latest}")
    datasets.append(project.version(latest).download("yolov8"))

for d in datasets:
    print(d.location)

In [ ]:
# Pool the downloads into one dataset. Class indices are remapped BY NAME, so this stays correct even if a project
# reorders its classes - the thing you can't see going wrong by eye, and the thing that would quietly teach the
# model to call a PSU a GPU.
import shutil, yaml
from collections import Counter
from pathlib import Path

MERGED = Path("/content/merged")

# Other projects' names for the first project's classes (compared in lower case). "CPU", "GPU" and "PSU" only
# differ in case, so they need no entry.
CLASS_ALIASES = {
    "cpu_fan": "cpu_cooler",
    "disk": "disk_drive",
    "mainboard": "motherboard",
    "ram": "ram_stick",
}

def canon(name):
    n = str(name).strip().lower()
    return CLASS_ALIASES.get(n, n)

def class_names(location):
    with open(Path(location) / "data.yaml") as f:
        names = yaml.safe_load(f)["names"]
    if isinstance(names, dict):                  # some exports use {0: 'cpu', 1: ...}
        names = [names[k] for k in sorted(names, key=int)]
    return [canon(n) for n in names]

# The first dataset sets the order; anything new is appended rather than renumbered.
canonical = class_names(datasets[0].location)
APP_CLASSES = list(canonical)
for d in datasets[1:]:
    for n in class_names(d.location):
        if n not in canonical:
            canonical.append(n)
if canonical != APP_CLASSES:
    print("NOTE: new classes", canonical[len(APP_CLASSES):], "- the app needs the new labels file, and each new name")
    print("      needs a line in Scripts/Data/MlLabelMap.cs (or an alias here, if it is just another spelling).")
print(len(canonical), "classes:", canonical)

if MERGED.exists():
    shutil.rmtree(MERGED)

counts, boxes = Counter(), [Counter() for _ in datasets]
for i, d in enumerate(datasets):
    remap = {j: canonical.index(n) for j, n in enumerate(class_names(d.location))}
    for split in ("train", "valid", "test"):
        src = Path(d.location) / split
        if not (src / "images").is_dir():
            continue
        (MERGED / split / "images").mkdir(parents=True, exist_ok=True)
        (MERGED / split / "labels").mkdir(parents=True, exist_ok=True)
        for image in (src / "images").iterdir():
            if image.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp"):
                continue
            # Projects share file names, so tag each copy with the dataset it came from.
            shutil.copy(image, MERGED / split / "images" / f"d{i}_{image.name}")
            label, lines = src / "labels" / (image.stem + ".txt"), []
            if label.exists():
                for line in label.read_text().splitlines():
                    parts = line.split()
                    if not parts:
                        continue
                    mapped = remap.get(int(parts[0]))
                    if mapped is None:
                        continue
                    parts[0] = str(mapped)
                    lines.append(" ".join(parts))
                    boxes[i][canonical[mapped]] += 1
            (MERGED / split / "labels" / f"d{i}_{image.stem}.txt").write_text("\n".join(lines))
            counts[split] += 1

has_test = (MERGED / "test" / "images").is_dir()
def write_yaml(path, val):
    with open(path, "w") as f:
        yaml.safe_dump({"path": str(MERGED), "train": "train/images", "val": val,
                        "test": "test/images" if has_test else "valid/images",
                        "nc": len(canonical), "names": canonical}, f, sort_keys=False)

DATA = str(MERGED / "data.yaml")
write_yaml(DATA, "valid/images")

# One validation list per source, so each dataset can be scored on its own (see the comparison cell).
SOURCE_DATA = {}
for i, (workspace, name) in enumerate(SOURCES):
    images = sorted(str(p) for p in (MERGED / "valid" / "images").glob(f"d{i}_*"))
    if not images:
        continue
    (MERGED / f"valid_d{i}.txt").write_text("\n".join(images))
    SOURCE_DATA[name] = str(MERGED / f"data_d{i}.yaml")
    write_yaml(SOURCE_DATA[name], f"valid_d{i}.txt")

print("merged images per split:", dict(counts))
print("\nlabelled boxes per class, per source (a 0 means that source never labels it):")
print(f"{'class':14}" + "".join(f"{n[:18]:>20}" for _, n in SOURCES))
for c in canonical:
    print(f"{c:14}" + "".join(f"{boxes[i][c]:>20}" for i in range(len(datasets))))

In [ ]:
# The class ORDER here is what the app must use. Writing the labels file from the merged class list removes any
# chance of getting it wrong by hand. With the aliases above it is the same 12 names the app already has.
with open("labels_pcparts_roboflow.txt", "w") as f:
    f.write("\n".join(canonical) + "\n")

print(len(canonical), "classes, in model order:")
for i, n in enumerate(canonical):
    print(f"{i:2d}. {n}")

In [ ]:
# MODEL: yolo11s is ~3x the compute of yolo11n and clearly more accurate. The app runs the model about 5 times a
# second, which a mid-range phone manages with yolo11s; if the scanner feels slow on yours, switch to yolo11n.pt.
#
# The augmentation settings matter more than the epoch count: much of the data is clean, well-lit photos, while
# the app sees a phone camera at an angle in a bedroom. degrees/scale/perspective and the brightness jitter
# manufacture that variety instead of you photographing it.
#
# SESSION DROPPED? Re-run the cells above this one (not this one), then run instead:
#   !yolo detect train resume model={RUN}/weights/last.pt
MODEL = "yolo11s.pt"
PROJECT = f"{OUT}/runs"
RUN = f"{PROJECT}/pcparts"

!yolo detect train model={MODEL} data={DATA} project={PROJECT} name=pcparts exist_ok=True \
    imgsz=640 epochs=100 batch=16 patience=25 cos_lr=True plots=True \
    degrees=15 scale=0.6 perspective=0.0005 hsv_v=0.5 erasing=0.2 close_mosaic=10

In [ ]:
# Overall scores on the pooled validation set. mAP50 above ~0.8 is comfortable. Watch the confusion matrix for the
# look-alike classes (ram_slot vs ram_stick, gpu vs gpu_slot, motherboard vs everything on it) - those pairs are
# what make the scanner name the wrong part on the phone.
!yolo detect val model={RUN}/weights/best.pt data={DATA} project={PROJECT} name=val exist_ok=True

from IPython.display import Image, display
display(Image(f"{RUN}/results.png", width=900))
display(Image(f"{RUN}/confusion_matrix_normalized.png", width=700))

In [ ]:
# DID IT ACTUALLY GET BETTER? Score the previous model and the new one on the SAME images, one source at a time.
# mAP from two different validation sets is not comparable, so this is the only comparison that means anything.
#   - james-manalili / cas: the data the previous model was trained on. The new one should at least hold its own.
#   - mynhungs: the single-part photos. This is where the new one should be clearly ahead.
#
# PREVIOUS: upload the old model from Assets/_Project/ML/Models/best.onnx to Colab (Files pane) or to Drive.
# Ultralytics can score an .onnx directly. Set it to None to skip the comparison.
import os
from ultralytics import YOLO

PREVIOUS = f"{OUT}/previous_best.onnx"

models = {"new": f"{RUN}/weights/best.pt"}
if PREVIOUS and os.path.exists(PREVIOUS):
    models["previous"] = PREVIOUS
else:
    print(f"No previous model at {PREVIOUS} - reporting the new model only.\n")

print(f"{'validation set':28}" + "".join(f"{m:>22}" for m in models))
for source, data in SOURCE_DATA.items():
    row = f"{source:28}"
    for name, weights in models.items():
        r = YOLO(weights, task="detect").val(data=data, imgsz=640, batch=1 if weights.endswith(".onnx") else 16,
                                             verbose=False, plots=False)
        row += f"   mAP50 {r.box.map50:.3f} ({r.box.map:.3f})"
    print(row)
print("\n(mAP50, with mAP50-95 in brackets.) Ship the new model only if it is ahead where it matters to you.")

In [ ]:
# opset 17 is inside the range the app's Inference Engine supports (7-25).
# Do NOT add nms=True: the app decodes and runs NMS itself.
!yolo export model={RUN}/weights/best.pt format=onnx imgsz=640 opset=17 simplify=True

import shutil
shutil.copy("labels_pcparts_roboflow.txt", OUT)
print("model and labels are in", f"{RUN}/weights", "and", OUT)

In [ ]:
from google.colab import files

files.download(f"{RUN}/weights/best.onnx")
files.download("labels_pcparts_roboflow.txt")

## Back in Unity

1. Keep a copy of the old `Assets/_Project/ML/Models/best.onnx` (rename it, e.g. `best_v1.onnx.bak`, outside
   `Assets`), then copy the new `best.onnx` and `labels_pcparts_roboflow.txt` over the old pair. With the class
   aliases the list is unchanged, so nothing else in the project needs touching.
2. Run **BuildAR ▸ Debug ▸ Check ML Model**. It runs the model once and prints the input size, the output layout,
   the class count, which labels file matches, and what each class maps to. Fix anything it flags.
3. Open `BuildAR_ARScanner`, select **Detectors ▸ InferenceComponentDetector** and confirm **Model Asset**,
   **Labels File** and **Input Size** (640).
4. Build to the phone and scan your real parts. If the frame rate drops noticeably, lower **Inferences Per
   Second** on the detector, or retrain with `MODEL = "yolo11n.pt"`.

Credit all three datasets (CC BY 4.0): *PC parts Dataset, James Manalili, Roboflow Universe*; *PC Parts Detection
Dataset, CAS PC Project, Roboflow Universe*; *PC-DETECTION Dataset, MyNhungs Workspace, Roboflow Universe.*